# Phase 3: Candidate Matching Engine + Explainable AI

This notebook picks up where `NLP_Cleaning_and_extraction.ipynb` left off.

**Pipeline position** (from `new_proposed_pipeline`):
```
Feature Generation  →  Candidate Matching Engine  →  Suitability Prediction  →  Explainable AI Module
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                              THIS NOTEBOOK
```

### ⚠️ Known limitation -- please read before interpreting results below
`match_score` / `match_label` in this dataset are **not human-annotated ground truth**.
Checking the data shows `match_label` is a *deterministic* function of `match_score`
(hard cutoffs at 0.40 and 0.75, zero exceptions across all 500 rows) -- meaning the
label was generated by a rule/script, not by a recruiter. A simple linear regression
using our 4 extracted features only explains ~47% of the variance in `match_score`,
confirming some of what generated the label is invisible to us.

**What this means:** we are teaching a model to reconstruct a synthetic proxy label
from interpretable features. That's still a legitimate exercise for this project --
the explainability layer works regardless of how the label was produced -- but we
should not claim the model has learned "true" recruiter judgment.


## 1. Setup

In [ ]:
# If running in Colab: upload pipeline.py and resumeJD2_pairs.csv when prompted
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    pass  # not running in Colab, assume files are already local

!pip install rapidfuzz shap sentence-transformers scikit-learn pdfplumber -q


In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

import pipeline as p   # our consolidated, bug-fixed extraction module

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

import shap

RANDOM_STATE = 42


## 2. Rebuild the feature set with the fixed pipeline

We regenerate every feature from the RAW resume/JD pairs using `pipeline.py`, rather
than reusing the old CSV as-is -- this brings in every fix made to the extraction
step (the symbol-stripping bug, the expanded ~280-term skill dictionary, and fuzzy
matching for typos/variants).

In [ ]:
raw_df = pd.read_csv("resumeJD2_pairs.csv")

# build_dataset() runs the FULL pipeline: cleaning, skills (exact + fuzzy),
# experience, education, domain, and semantic similarity (Sentence-BERT).
# This downloads the all-MiniLM-L6-v2 model on first run.
feature_df = p.build_dataset(raw_df)

print(feature_df.shape)
feature_df.head(3)


## 3. Prepare features for modeling

Two things worth calling out here, both explained in earlier project discussion:

1. **`experience_match_score` can be NaN** when no years-of-experience phrase was
   found in the resume or JD. We fill it using the **training fold's median only**
   (never the full dataset's median) to avoid leaking test-set information into
   training.
2. We keep `experience_mentioned` as its own feature so "we don't know" is preserved
   as a signal rather than silently erased by imputation.

In [ ]:
FEATURES = [
    "skill_match_score",
    "matched_skill_count",
    "missing_skill_count",
    "experience_match_score",
    "resume_experience_mentioned",
    "jd_experience_mentioned",
    "semantic_similarity_score",
    "domain_match",
]

X = feature_df[FEATURES].copy()
y = feature_df["match_label"]

print("NaN counts before imputation:")
print(X.isna().sum())


## 4. Compare Random Forest vs Logistic Regression (stratified 5-fold CV)

We don't pick a model by assumption -- we compare both properly, with the same
folds, and let the numbers decide. Experience imputation happens **inside** each
fold using only that fold's training data, to avoid leakage.

In [ ]:
def run_cv(model_builder, X, y, n_splits=5):
    """Manual CV loop so we can impute experience_match_score per-fold
    using only the training split's median (no leakage)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    all_true, all_pred = [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        train_median = X_train["experience_match_score"].median()
        X_train["experience_match_score"] = p.impute_experience_score(
            X_train["experience_match_score"], train_median)
        X_test["experience_match_score"] = p.impute_experience_score(
            X_test["experience_match_score"], train_median)  # SAME value, no leakage

        model = model_builder()
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        all_true.extend(y_test)
        all_pred.extend(preds)

    return all_true, all_pred


rf_builder = lambda: RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=5,
    random_state=RANDOM_STATE, class_weight="balanced")

lr_builder = lambda: Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

rf_true, rf_pred = run_cv(rf_builder, X, y)
lr_true, lr_pred = run_cv(lr_builder, X, y)

print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(rf_true, rf_pred):.3f}")
print(classification_report(rf_true, rf_pred))

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(lr_true, lr_pred):.3f}")
print(classification_report(lr_true, lr_pred))


**Interpreting this:** both models land in a similar place. Notice "partial match"
has the weakest recall for both -- this isn't a model-choice problem, it's because
`match_label` is `match_score` sliced into bands, and "partial match" is squeezed
between two cutoffs (0.40 and 0.75) instead of one, so it's more sensitive to the
~53% of `match_score`'s variance our features can't see.

**We proceed with Random Forest** -- not because it's meaingfully more accurate,
but because it pairs with `shap.TreeExplainer` (exact, fast), which is central to
this project's explainability goal.

## 5. Confidence-based human review

Rather than forcing every prediction into a hard bucket, we use the model's own
`predict_proba()` confidence. Below a threshold, we flag the candidate for manual
review instead of guessing -- directly implementing the "Human-AI Collaborative
Decision Support" feature from the original proposal.

In [ ]:
CONFIDENCE_THRESHOLD = 0.55

def cv_with_confidence(model_builder, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    results = []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        train_median = X_train["experience_match_score"].median()
        X_train["experience_match_score"] = p.impute_experience_score(X_train["experience_match_score"], train_median)
        X_test["experience_match_score"] = p.impute_experience_score(X_test["experience_match_score"], train_median)

        model = model_builder()
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)
        pred = model.classes_[np.argmax(proba, axis=1)]
        conf = np.max(proba, axis=1)

        for i, idx in enumerate(test_idx):
            results.append({"true": y_test.iloc[i], "pred": pred[i], "confidence": conf[i]})

    return pd.DataFrame(results)


cv_results = cv_with_confidence(rf_builder, X, y)
cv_results["needs_human_review"] = cv_results["confidence"] < CONFIDENCE_THRESHOLD

pct_flagged = cv_results["needs_human_review"].mean() * 100
confident_subset = cv_results[~cv_results["needs_human_review"]]
acc_confident = accuracy_score(confident_subset["true"], confident_subset["pred"])

print(f"Candidates flagged for human review: {pct_flagged:.1f}%")
print(f"Accuracy on confident (non-flagged) predictions: {acc_confident:.3f}")
print()
print(classification_report(confident_subset["true"], confident_subset["pred"]))


## 6. Train the final production model

We now train on the FULL dataset (no held-out split) since cross-validation above
already gave us an honest estimate of real-world performance. This is the model
that gets saved and deployed.

In [ ]:
X_full = X.copy()
full_median = X_full["experience_match_score"].median()
X_full["experience_match_score"] = p.impute_experience_score(X_full["experience_match_score"], full_median)

final_model = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=5,
    random_state=RANDOM_STATE, class_weight="balanced")
final_model.fit(X_full, y)

joblib.dump(final_model, "rf_model.joblib")
joblib.dump({"experience_median": full_median, "confidence_threshold": CONFIDENCE_THRESHOLD,
             "features": FEATURES}, "model_config.joblib")

print("Saved rf_model.joblib and model_config.joblib")


## 7. Explainable AI Module (SHAP)

This is the "Explainable AI Module" block from the proposed architecture. We use
`shap.TreeExplainer`, which is exact (not approximate) for tree models like ours.

In [ ]:
explainer = shap.TreeExplainer(final_model)

# Global feature importance across the whole dataset
shap_values_all = explainer.shap_values(X_full)

shap.summary_plot(shap_values_all[:, :, list(final_model.classes_).index("match")],
                   X_full, plot_type="bar", show=False)
plt.title("Feature importance for predicting 'Match'")
plt.tight_layout()
plt.show()


### Per-candidate explanation

This turns SHAP's numeric output into the natural-language report format from the
project proposal (Matched Skills / Missing Skills / Prediction / Reasons).

In [ ]:
FEATURE_LABELS = {
    "skill_match_score": "Skill Match",
    "matched_skill_count": "Number of Matched Skills",
    "missing_skill_count": "Number of Missing Skills",
    "experience_match_score": "Experience Match",
    "resume_experience_mentioned": "Resume States Experience",
    "jd_experience_mentioned": "JD States Required Experience",
    "semantic_similarity_score": "Overall Semantic Fit",
    "domain_match": "Domain Match",
}


def generate_explanation_report(row_features: dict, matched_skills: list, missing_skills: list,
                                 model=final_model, explainer=explainer,
                                 confidence_threshold: float = CONFIDENCE_THRESHOLD) -> dict:
    """
    Takes one candidate's feature dict (matching FEATURES) plus their matched/missing
    skill lists, and returns a full explanation report -- this is the function your
    API endpoint calls after extract_features().
    """
    X_row = pd.DataFrame([row_features])[FEATURES]

    proba = model.predict_proba(X_row)[0]
    pred_class = model.classes_[np.argmax(proba)]
    confidence = float(np.max(proba))
    needs_review = confidence < confidence_threshold

    sv = explainer.shap_values(X_row)
    class_idx = list(model.classes_).index(pred_class)
    contributions = sv[0, :, class_idx]
    impact = sorted(zip(FEATURES, contributions), key=lambda t: -abs(t[1]))

    reasons = []
    for feat, contrib in impact[:4]:
        direction = "supported" if contrib > 0 else "worked against"
        reasons.append({
            "factor": FEATURE_LABELS[feat],
            "value": round(float(X_row[feat].values[0]), 2),
            "direction": direction,
            "impact": round(float(contrib), 3),
        })

    return {
        "prediction": pred_class,
        "confidence": round(confidence, 2),
        "needs_human_review": needs_review,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "reasons": reasons,
    }


def print_report(report: dict):
    print(f"Prediction: {report['prediction'].upper()}  (confidence: {report['confidence']:.0%})")
    if report["needs_human_review"]:
        print("\u26a0\ufe0f  Low confidence -- flagged for human review")
    print()
    print("Matched Skills:", ", ".join(report["matched_skills"]) or "None")
    print("Missing Skills:", ", ".join(report["missing_skills"]) or "None")
    print()
    print("Why this prediction:")
    for r in report["reasons"]:
        print(f"  - {r['factor']} ({r['value']}) {r['direction']} this prediction (impact: {r['impact']:+.3f})")


## 8. End-to-end test: new resume + JD → prediction → explanation

In [ ]:
resume_text = """
John Doe
Data Scientist with 5 years of experience in machine learning and Python.
Skills: Python, SQL, Machine Learning, TensorFlow, AWS, Docker
Education: B.Tech in Computer Science
"""

jd_text = """
Job Title: Machine Learning Engineer
Experience: 4 years required
Required Skills: Python, TensorFlow, AWS, Kubernetes, SQL
"""

extracted = p.extract_features(resume_text, jd_text)

row_features = {
    "skill_match_score": extracted["skill_match_score"],
    "matched_skill_count": extracted["matched_skill_count"],
    "missing_skill_count": extracted["missing_skill_count"],
    "experience_match_score": (extracted["experience_match_score"]
                                if not pd.isna(extracted["experience_match_score"])
                                else full_median),
    "resume_experience_mentioned": extracted["resume_experience_mentioned"],
    "jd_experience_mentioned": extracted["jd_experience_mentioned"],
    "semantic_similarity_score": extracted["semantic_similarity_score"],
    "domain_match": extracted["domain_match"],
}

report = generate_explanation_report(row_features, extracted["matched_skills"], extracted["missing_skills"])
print_report(report)


### Same thing, starting from an actual PDF file

In [ ]:
# resume_pdf_features = p.extract_features_from_pdf("resume.pdf", jd_text)
# (uncomment above with a real uploaded resume.pdf; wrap in try/except ScannedPDFError/CorruptPDFError
#  from pipeline.py to handle bad uploads gracefully)


## 9. Known Limitations (for the report)

1. **Synthetic label**: `match_score`/`match_label` were not human-annotated; our 4-6
   features explain ~47% of `match_score`'s variance. Framed correctly, we're learning
   to reconstruct a proxy score, not "true" recruiter judgment.
2. **"Partial Match" is the hardest class** for exactly this reason -- it's bounded by
   two cutoffs (0.40, 0.75) instead of one, so it's most sensitive to the unexplained
   variance. Addressed via confidence-based human review rather than forcing accuracy
   we can't honestly achieve.
3. **Skill dictionary is closed-vocabulary** (~280 terms + fuzzy matching for typos/
   variants). Brand-new tools not in the dictionary won't be detected.
4. **Scanned/image PDFs** aren't supported (would need OCR) -- handled with a clear
   error (`ScannedPDFError`) rather than a silent failure or crash.
5. **Dataset size (500 rows)** limits how much a more complex model could help --
   this is why we didn't reach for deep learning or heavy hyperparameter search.